# Fields

In [ ]:
import mefikit as mf
import numpy as np
import pyvista as pv

pv.set_plot_theme("dark")
pv.set_jupyter_backend("static")

## Field expressions


FieldExpr are composition of floats and mf.sel.field("fieldname") or custom fields :
- field("toto") * field("tata")
- field("toto") + field("tata")
- field("toto") - field("tata")
- field("toto") / field("tata")
- field("toto") ** field("tata")
- field("toto").dot(field("tata"))
- field("toto") @ field("tata")
- sin(field("toto"))
- cos(field("toto"))
- abs(field("toto"))
- log(field("toto"))
- log10(field("toto"))
- exp(field("toto"))
- field("toto")[0]

In [ ]:
x = np.logspace(-4, 0.0, 4)
z = np.linspace(0.0, 0.1, 3)
mesh2 = mf.build_cmesh(x, x, z)

In [ ]:
# mesh2.measure_update()

In [ ]:
mesh2.to_pyvista().plot(show_edges=True)

In [ ]:
mesh2.fields["Measure"] = mf.M

In [ ]:
pv.set_jupyter_backend("trame")
mesh2.to_pyvista().shrink(0.8).plot()

In [ ]:
print(mesh2)

In [ ]:
mesh2.fields["toto"] = mf.X

In [ ]:
mesh2.to_pyvista().plot(show_edges=True)

## The fields mapping

Fields live in a dict-like mapping on the mesh, keyed by name. Each entry is a handle (`FieldRef`) to read values, reduce them, or write through selectors.

In [ ]:
# List and look up fields by name.
print(mesh2.fields.keys())
mes = mesh2.fields["Measure"]
print("shape:", mes.shape, "| elements:", len(mes))

In [ ]:
# Bulk export as {etype: array} (or a single array via `.numpy()` when the
# mesh has one element type).
vals = mes.values()["HEX8"]
assert np.allclose(vals.ravel(), np.asarray(mes.numpy()).ravel())

In [ ]:
# Whole-domain reductions over every element carrying the field.
print(mes.min(), mes.max(), mes.mean())

# Regional reductions: combine a lazy selection with any field expression,
# including plain existing field names as strings.
rect = mf.sel.bbox([0.25, 0.25, 0.0], [0.7, 0.7, 0.1])
zone = mesh2.select(rect)
print(zone.mean("Measure"), zone.max(mf.Field("Measure") * 4))

In [ ]:
# Writes accept scalars, arrays, field expressions or existing field names,
# targeted by wildcards (`...`) or selectors.
mesh2.fields["Scratch"] = 0.0  # create by broadcast
mesh2.fields["Scratch"][...] = "Measure"  # copy an existing field

sel = mf.sel.bbox([0.0, 0.0, 0.0], [0.3, 1.0, 0.1])
mesh2.fields["Scratch"][sel] = mf.Field("Measure") * 2  # scaled sub-region

sel2 = mf.sel.sphere(center=[0.5, 0.5, 0.05], r2=0.25)
m = mesh2.select(sel2).mean("Scratch")
mesh2.fields["Scratch"][sel2] = m

In [ ]:
pt = pv.Plotter()
pvm = mesh2.to_pyvista()
pvm.active_scalars_name = "Scratch"
pt.add_mesh(pvm)
pt.show()

In [ ]:
del mesh2.fields["Scratch"]  # remove it
print(mesh2.fields)

## Field operations evaluation

In [ ]:
m = mf.Field("Measure")
mesh2.fields["4 * M2"] = m * m * 4.0

In [ ]:
mesh2.to_pyvista().plot()

In [ ]:
mesh2.fields

In [ ]:
m2 = mf.Field("4 * M2")
mesh2.eval(m2 - 4.0 * m.square())["HEX8"]

## How does it work ?

In [ ]:
print(4.0 * m * m)

## Why is it awesome ?

This enables two patterns :
- reusability and composition of filters
- evaluation optimizations, some selection filters are evaluated in parallel, some are evaluated first if they are discriminant

# Field to Selection

Fields can be converted to threshold selections :

In [ ]:
th = (m > 5.0e-7) & (m < 3.0e-6)

In [ ]:
m2sel = mesh2.select(th).to_mesh()
pvm2: pv.UnstructuredGrid = m2sel.to_pyvista()
pvm2.active_scalars_name = "Measure"
pvm2.plot()

Those threshold selections can be combined with other selections.

In [ ]:
r = mf.sel.bbox([0.25, 0.25, 0.0], [0.7, 0.7, 0.1])
c = mf.sel.sphere([0.875, 0.875, 0.05], 0.05)

In [ ]:
mesh2.select((m2 > 2e-11) - r - c).to_mesh().to_pyvista().plot()